# SPXW data preparation

This notebook builds the option panel used by the hedging experiments from
OptionMetrics IvyDB daily quotes, S&P 500 closes and one-year Treasury yields.
It covers 2007–2024 and retains PM-settled calls. See
[data/README.md](../data/README.md) for data access, the input schema and setup
commands. OptionMetrics data are supplied by the user under their own license.

Run all cells in order. The outputs are
`data/SPX_2007_2024_clean_SPXW.parquet` and
`new-reward/data/1yr_treasury.csv` under the repository root.

## 1. Input and output paths

The default inputs are `data/raw/SPX0025.parquet`, `data/raw/DGS1.csv`
and `data/raw/spot.csv`. If the spot CSV is absent, the notebook downloads
S&P 500 closes. Input-path environment variables are optional overrides;
see the data guide when storing inputs elsewhere.


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

def _find_repository_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'ctc_runtime.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the conformal-trust-control repository.')

WORKSPACE = Path(os.environ.get('CTC_WORKSPACE', _find_repository_root())).expanduser().resolve()
DATA_PATH = Path(os.environ.get('SPX0025_PATH', WORKSPACE / 'data/raw/SPX0025.parquet')).expanduser().resolve()
TREASURY_PATH = Path(os.environ.get('CTC_TREASURY_PATH', WORKSPACE / 'data/raw/DGS1.csv')).expanduser().resolve()
OUTPUT_PATH = WORKSPACE / 'data/SPX_2007_2024_clean_SPXW.parquet'
RATE_OUTPUT_PATH = WORKSPACE / 'new-reward/data/1yr_treasury.csv'
for path in (DATA_PATH, TREASURY_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

print('Option quotes:', DATA_PATH)
print('Treasury series:', TREASURY_PATH)
print('Cleaned panel:', OUTPUT_PATH)


## 2. Load daily option quotes

The date filter is applied while reading parquet. Numeric columns use the
same reduced-precision types as the experiment's cleaning procedure.

In [ ]:
USE_COLS = ['date', 'exdate', 'cp_flag', 'strike_price',
            'best_bid', 'best_offer', 'volume', 'open_interest',
            'impl_volatility', 'delta', 'gamma', 'vega', 'theta',
            'forward_price', 'root', 'am_settlement']
dataset = ds.dataset(DATA_PATH, format='parquet')
missing = set(USE_COLS).difference(dataset.schema.names)
if missing:
    raise ValueError(f'Option input is missing columns: {sorted(missing)}')
date_filter = ((ds.field('date') >= pd.Timestamp('2007-01-01')) &
               (ds.field('date') <= pd.Timestamp('2024-12-31')))
df = dataset.to_table(columns=USE_COLS, filter=date_filter).to_pandas()
if df.empty:
    raise ValueError('No option quotes found in 2007–2024.')

float_cols = ['best_bid', 'best_offer', 'impl_volatility',
              'delta', 'gamma', 'vega', 'theta', 'forward_price']
int_cols = ['volume', 'open_interest', 'strike_price', 'am_settlement']
for name in float_cols:
    df[name] = pd.to_numeric(df[name], downcast='float')
for name in int_cols:
    df[name] = pd.to_numeric(df[name], downcast='integer')
df['cp_flag'] = df['cp_flag'].astype('category')
df['root'] = df['root'].astype('category')
print(f'Loaded {len(df):,} quotes from {df["date"].min()} to {df["date"].max()}')


## 3. Settlement and quote filters

Keep PM-settled quotes, positive non-crossed bid/ask prices, relative spreads
at most 50% of the midpoint and finite positive implied volatilities. Convert
strikes from the vendor's thousandths of an index point to index points.

In [ ]:
def retain(frame, mask, label):
    filtered = frame.loc[mask].reset_index(drop=True)
    print(f'{label}: {len(frame):,} -> {len(filtered):,} rows')
    return filtered

keep_cols = ['date', 'exdate', 'cp_flag', 'strike_price', 'best_bid',
             'best_offer', 'impl_volatility', 'delta', 'volume',
             'open_interest', 'forward_price', 'am_settlement']
df = df[keep_cols]
df = retain(df, df['am_settlement'].eq(0), 'PM settlement')
df = df.drop(columns='am_settlement')
df = retain(df, (df['best_bid'] > 0) & (df['best_offer'] > 0), 'Positive quotes')
df['strike_price'] = pd.to_numeric(df['strike_price'] / 1000.0, downcast='float')
df = retain(df, df['best_offer'] >= df['best_bid'], 'Non-crossed quotes')
mid = (df['best_bid'] + df['best_offer']) / 2.0
spread = df['best_offer'] - df['best_bid']
df = retain(df, spread <= 0.5 * mid, 'Relative spread')
df = retain(df, np.isfinite(df['impl_volatility']) & (df['impl_volatility'] > 0),
            'Positive implied volatility')


## 4. Join S&P 500 closes

A local CSV should contain one `date,spot` row per trading day. The download
uses unadjusted `^GSPC` closes. Quotes without a matching close are removed.

In [ ]:
spot_path = os.environ.get('CTC_SPOT_CSV')
if spot_path is None and (WORKSPACE / 'data/raw/spot.csv').is_file():
    spot_path = str(WORKSPACE / 'data/raw/spot.csv')
if spot_path:
    spx = pd.read_csv(Path(spot_path).expanduser(), usecols=['date', 'spot'])
else:
    import yfinance as yf
    spx = yf.download('^GSPC', start='2006-12-31', end='2025-01-02',
                      auto_adjust=False, progress=False)[['Close']].reset_index()
    spx.columns = ['date', 'spot']
spx['date'] = pd.to_datetime(spx['date']).dt.strftime('%Y-%m-%d')
spx['spot'] = pd.to_numeric(spx['spot'], downcast='float')
if spx['date'].duplicated().any():
    raise ValueError('Spot input must have one row per date.')
if not (np.isfinite(spx['spot']) & (spx['spot'] > 0)).all():
    raise ValueError('Spot prices must be finite and positive.')
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
before = len(df)
df = df.merge(spx, on='date', how='inner', validate='many_to_one')
print(f'Matched index closes: {before:,} -> {len(df):,} rows')


## 5. Contract filters

Require prices at least equal to intrinsic value, 10–90 calendar days to
expiry and spot/strike in [0.83, 1.17]. Keep calls after applying these filters.

In [ ]:
mid = (df['best_bid'] + df['best_offer']) / 2.0
intrinsic = np.where(df['cp_flag'].eq('C'),
                     np.maximum(df['spot'] - df['strike_price'], 0.0),
                     np.maximum(df['strike_price'] - df['spot'], 0.0))
df = retain(df, mid >= intrinsic, 'Intrinsic-value bound')
df['date'] = pd.to_datetime(df['date'])
df['exdate'] = pd.to_datetime(df['exdate'])
df['dtm'] = (df['exdate'] - df['date']).dt.days.astype('int16')
df = retain(df, df['dtm'].between(10, 90), 'Days to expiry')
df['S_over_K'] = (df['spot'] / df['strike_price']).astype('float32')
df = retain(df, df['S_over_K'].between(0.83, 1.17), 'Moneyness')
df = retain(df, df['cp_flag'].eq('C'), 'Calls')
if df.empty:
    raise ValueError('No quotes remain after filtering; check input units and contract selection.')


## 6. Join Treasury yields

The experiment uses decimal yields in a `Date,1y` CSV. A FRED DGS1 download
uses percent and is converted below. Missing observations are carried forward
on the option-date grid; the first retained quote date must have a rate.

In [ ]:
treasury = pd.read_csv(TREASURY_PATH)
if {'Date', '1y'}.issubset(treasury.columns):
    rates = treasury[['Date', '1y']].rename(columns={'Date': 'date', '1y': 'r'})
elif 'DGS1' in treasury.columns:
    date_col = next((c for c in ('observation_date', 'DATE', 'Date', 'date')
                     if c in treasury.columns), None)
    if date_col is None:
        raise ValueError('FRED CSV must contain a date column and DGS1.')
    rates = treasury[[date_col, 'DGS1']].rename(columns={date_col: 'date', 'DGS1': 'r'})
    rates['r'] = pd.to_numeric(rates['r'], errors='coerce') / 100.0
else:
    raise ValueError('Treasury CSV must have Date,1y (decimals) or observation_date,DGS1 (percent).')
rates['date'] = pd.to_datetime(rates['date'], errors='coerce')
rates['r'] = pd.to_numeric(rates['r'], errors='coerce')
rates = rates.dropna().sort_values('date').reset_index(drop=True)
if rates['date'].duplicated().any():
    raise ValueError('Treasury input must have one rate per date.')
if not np.isfinite(rates['r']).all():
    raise ValueError('Treasury rates must be finite.')

quote_dates = pd.DataFrame({'date': sorted(df['date'].unique())})
joined_rates = quote_dates.merge(rates, on='date', how='left').ffill()
if joined_rates['r'].isna().any():
    raise ValueError('Treasury input has no rate on the first retained quote date.')
joined_rates['r'] = pd.to_numeric(joined_rates['r'], downcast='float')
df = df.merge(joined_rates, on='date', how='left', validate='many_to_one')
print(f'Treasury yields: {df["r"].min():.4f} to {df["r"].max():.4f}')


## 7. Build the environment input

The environment expects bid/ask fields for the underlying, so both are set
to the daily index close. The `09:31:00` timestamp is a formatting convention
for this interface, not the quote observation time. `nbr_next_steps` counts
later observations within each expiration/strike path.

In [ ]:
edh = pd.DataFrame({
    'quote_datetime': df['date'].dt.strftime('%Y-%m-%d') + ' 09:31:00',
    'ticker': 'SPX',
    'expiration': df['exdate'].dt.strftime('%Y-%m-%d'),
    'strike': df['strike_price'].astype('float32'),
    'underlying_bid': df['spot'],
    'underlying_ask': df['spot'],
    'bid': df['best_bid'],
    'ask': df['best_offer'],
    'date': df['date'],
    'dtm': df['dtm'],
    'impl_volatility': df['impl_volatility'],
    'delta': df['delta'],
    'r': df['r'],
})
edh = edh.sort_values(['expiration', 'strike', 'date']).reset_index(drop=True)
groups = edh.groupby(['expiration', 'strike'], sort=False)
edh['nbr_next_steps'] = (groups['date'].transform('count')
                         - groups.cumcount() - 1).astype('int16')
print(f'Cleaned panel: {len(edh):,} rows')
print(f'Contract paths: {groups.ngroups:,}')
print(f'Possible five-step episode starts: {(edh["nbr_next_steps"] >= 5).sum():,}')


## 8. Save the panel and rate series

The training notebooks read the parquet file from `data/` and construct the
expanding-window splits. The Treasury CSV is saved in the environment's
expected format. Keep both outputs in the experiment directory.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
RATE_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
edh.to_parquet(OUTPUT_PATH, index=False)
normalized_rates = rates.rename(columns={'date': 'Date', 'r': '1y'}).copy()
normalized_rates['Date'] = normalized_rates['Date'].dt.strftime('%Y-%m-%d')
normalized_rates.to_csv(RATE_OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)
print('Saved:', RATE_OUTPUT_PATH)
